# 01 · Data Understanding

Inventory of the 9 source files, schema, grain and relationships between entities.

> Data: Olist Brazilian E-Commerce public dataset (BRL). All narrative under charts is generated from the computed statistics.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160)
from src.analytics import eda
om, items, cm, daily = eda.load()

def show(f):
    if f.figure: display(Image(filename=str(ROOT / f.figure)))
    display(Markdown(f"**Business interpretation – {f.title}:** {f.interpretation}"))


## 1. Source inventory

In [2]:
eda.dataset_overview()

,dataset,rows,columns,memory_mb,null_cells_pct
0,category_translation,71,2,0.0,0.00
1,customers,99441,5,12.1,0.00
2,geolocation,1000163,5,57.6,0.00
3,order_items,112650,7,17.2,0.00
4,orders,99441,8,13.6,0.62
5,payments,103886,5,8.7,0.00
6,products,32951,9,3.9,0.83
7,reviews,99224,7,15.0,21.01
8,sellers,3095,4,0.3,0.00


## 2. Schemas and declared keys

In [3]:
from src.validation.schemas import SCHEMAS
pd.DataFrame([{'dataset': n, 'primary_key': ', '.join(s.primary_key) or '(none – many rows per zip)',
               'foreign_keys': '; '.join(f'{c} → {p}.{pc}' for c, (p, pc) in s.foreign_keys.items()),
               'description': s.description} for n, s in SCHEMAS.items()])

,dataset,primary_key,foreign_keys,description
0,customers,customer_id,,One row per customer_id. customer_id is order-...
1,orders,order_id,customer_id → customers.customer_id,Order header with lifecycle timestamps.
2,order_items,"order_id, order_item_id",order_id → orders.order_id; product_id → produ...,One row per item within an order (item sequenc...
3,payments,"order_id, payment_sequential",order_id → orders.order_id,One row per payment transaction of an order.
4,reviews,"review_id, order_id",order_id → orders.order_id,Customer satisfaction survey responses.
5,products,product_id,,"Product catalogue (category in Portuguese, phy..."
6,sellers,seller_id,,Marketplace sellers.
7,geolocation,(none – many rows per zip),,Lat/lng samples per 5-digit zip-code prefix (m...
8,category_translation,product_category_name,,Portuguese -> English category names.


## 3. Data dictionary (generated)

In [4]:
pd.read_csv('reports/data_quality/data_dictionary.csv').head(50)

,dataset,column,target_type,is_primary_key,foreign_key_to,null_pct,distinct_values,example,description
0,category_translation,product_category_name,string,True,NaN,0.000,71,beleza_saude,Product category (Portuguese).
1,category_translation,product_category_name_english,string,False,NaN,0.000,71,health_beauty,Category name translated to English.
2,customers,customer_id,string,True,NaN,0.000,99441,06b8999e2fba1a1fbc88172c00ba8bc7,Order-scoped customer key (one per order). FK ...
3,customers,customer_unique_id,string,False,NaN,0.000,96096,861eff4711a542e4b93843c6dd7febb0,Stable identifier of the real customer across ...
4,customers,customer_zip_code_prefix,string,False,NaN,0.000,14994,14409,First 5 digits of the customer zip code (strin...
5,customers,customer_city,string,False,NaN,0.000,4119,franca,"Customer city name (lower-case, Portuguese)."
6,customers,customer_state,string,False,NaN,0.000,27,SP,Customer state (2-letter Brazilian UF code).
7,geolocation,geolocation_zip_code_prefix,string,False,NaN,0.000,19015,01037,5-digit zip-code prefix.
8,geolocation,geolocation_lat,float,False,NaN,0.000,717360,-23.54562128115268,Latitude (decimal degrees).
9,geolocation,geolocation_lng,float,False,NaN,0.000,717613,-46.63929204800168,Longitude (decimal degrees).


## 4. Relationship cardinalities

In [5]:
eda.relationship_cardinality()

,relationship,avg_children,max_children,parents_without_children
0,orders → order_items,1.142,21,775
1,orders → payments,1.045,29,1
2,orders → reviews,1.006,3,768
3,customer_unique_id → customer_id,1.035,17,0


## 5. Grain decisions
* `customer_id` is **order-scoped**; the person is `customer_unique_id` → customer analytics use `customer_unique_id`.
* `review_id` is **not unique** – grain of reviews is (`review_id`, `order_id`).
* Payments can be split (`payment_sequential`) → order payment = sum over sequences.
* Geolocation has many points per zip prefix → aggregated to one row per prefix.

In [6]:
o = pd.read_parquet('data/staging/clean/orders.parquet')
print('Purchase range:', o.order_purchase_timestamp.min(), '→', o.order_purchase_timestamp.max())
o.order_status.value_counts().to_frame('orders')

Purchase range: 2016-09-04 21:15:19 → 2018-10-17 17:30:18


,orders
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [7]:
m = o.set_index('order_purchase_timestamp').resample('MS').order_id.count()
m.tail(6).to_frame('orders captured')  # note the collapse after Aug-2018 = extraction truncation

,orders captured
order_purchase_timestamp,
2018-05-01,6873
2018-06-01,6167
2018-07-01,6292
2018-08-01,6512
2018-09-01,16
2018-10-01,4
